# bashkit in Jupyter: async + custom_builtins compatibility

Jupyter already owns an asyncio event loop. This notebook checks that:
- `execute_sync()` works (creates its own private loop, no conflict)
- `await execute()` runs on Jupyter's caller loop
- Async callbacks reach the correct loop in both modes
- `custom_builtins` (BuiltinContext) work with sync and async callbacks
- ContextVar propagation works in Jupyter

In [1]:
import asyncio
import contextvars

from bashkit import Bash, BuiltinContext, ScriptedTool

print("bashkit imported OK")
print(f"Jupyter event loop: {asyncio.get_event_loop()}")

bashkit imported OK
Jupyter event loop: <_UnixSelectorEventLoop running=True closed=False debug=False>


## 1. Basic sync execution

`execute_sync()` creates a private internal loop — should not conflict with Jupyter's loop.

In [2]:
bash = Bash()
result = bash.execute_sync("echo 'hello from bashkit'")
assert result.exit_code == 0
assert result.stdout.strip() == "hello from bashkit"
print("sync execute OK:", repr(result.stdout.strip()))

sync execute OK: 'hello from bashkit'


## 2. Basic async execution

Jupyter lets you `await` at the top level — `execute()` should run on Jupyter's loop.

In [3]:
bash = Bash()
result = await bash.execute("echo 'hello async'")
assert result.exit_code == 0
assert result.stdout.strip() == "hello async"
print("async execute OK:", repr(result.stdout.strip()))

async execute OK: 'hello async'


## 3. custom_builtins — sync callback (BuiltinContext)

A Python function registered as a bash builtin via `custom_builtins=`.

In [4]:
def greet(ctx: BuiltinContext) -> str:
    name = ctx.argv[0] if ctx.argv else "world"
    return f"hello {name}\n"


bash = Bash(custom_builtins={"greet": greet})
result = bash.execute_sync("greet Jupiter")
assert result.exit_code == 0
assert result.stdout.strip() == "hello Jupiter"
print("sync builtin OK:", repr(result.stdout.strip()))

sync builtin OK: 'hello Jupiter'


## 4. custom_builtins — sync callback with pipeline

Builtin receiving stdin from a pipe.

In [5]:
def shout(ctx: BuiltinContext) -> str:
    text = ctx.stdin or ""
    return text.upper()


bash = Bash(custom_builtins={"shout": shout})
result = bash.execute_sync("echo 'hello pipe' | shout")
assert result.exit_code == 0
assert "HELLO PIPE" in result.stdout
print("sync builtin + pipe OK:", repr(result.stdout.strip()))

sync builtin + pipe OK: 'HELLO PIPE'


## 5. custom_builtins — async callback via execute_sync()

`execute_sync()` spawns its own private event loop to drive the async callback.
In Jupyter there's already a running loop — the private loop must be independent.

In [6]:
jupyter_loop = asyncio.get_event_loop()


async def async_greet(ctx: BuiltinContext) -> str:
    await asyncio.sleep(0)  # yield to loop
    name = ctx.argv[0] if ctx.argv else "world"
    return f"async-hello {name}\n"


bash = Bash(custom_builtins={"async_greet": async_greet})
result = bash.execute_sync("async_greet Jupyter")
assert result.exit_code == 0
assert result.stdout.strip() == "async-hello Jupyter"
print("async builtin via execute_sync OK:", repr(result.stdout.strip()))

async builtin via execute_sync OK: 'async-hello Jupyter'


## 6. custom_builtins — async callback via await execute()

With `await execute()`, async callbacks run on Jupyter's own event loop.

In [7]:
jupyter_loop = asyncio.get_running_loop()

captured_loop = None


async def loop_checker(ctx: BuiltinContext) -> str:
    global captured_loop
    captured_loop = asyncio.get_running_loop()
    return "checked\n"


bash = Bash(custom_builtins={"loop_checker": loop_checker})
result = await bash.execute("loop_checker")

assert result.exit_code == 0
assert captured_loop is jupyter_loop, "async callback must run on Jupyter's loop"
print("async builtin via await execute uses caller loop: OK")

async builtin via await execute uses caller loop: OK


## 7. ContextVar propagation — sync callback

ContextVars set in the Jupyter cell propagate into builtin callbacks.

In [8]:
request_id: contextvars.ContextVar[str] = contextvars.ContextVar("request_id")
request_id.set("jupyter-req-123")


def check_ctx(ctx: BuiltinContext) -> str:
    rid = request_id.get("MISSING")
    return f"req={rid}\n"


bash = Bash(custom_builtins={"check_ctx": check_ctx})
result = bash.execute_sync("check_ctx")
assert result.exit_code == 0
assert result.stdout.strip() == "req=jupyter-req-123"
print("ContextVar propagation (sync builtin) OK:", repr(result.stdout.strip()))

ContextVar propagation (sync builtin) OK: 'req=jupyter-req-123'


## 8. ContextVar propagation — async callback via await execute()

In [9]:
request_id.set("jupyter-req-456")


async def async_check_ctx(ctx: BuiltinContext) -> str:
    rid = request_id.get("MISSING")
    return f"req={rid}\n"


bash = Bash(custom_builtins={"async_check_ctx": async_check_ctx})
result = await bash.execute("async_check_ctx")
assert result.exit_code == 0
assert result.stdout.strip() == "req=jupyter-req-456"
print("ContextVar propagation (async builtin + await execute) OK:", repr(result.stdout.strip()))

ContextVar propagation (async builtin + await execute) OK: 'req=jupyter-req-456'


## 9. ScriptedTool with custom tools in Jupyter

`ScriptedTool.add_tool()` registers Python callbacks (both sync and async).

In [10]:
def reverse(params, stdin=None):
    text = params.get("text", stdin or "")
    return text[::-1] + "\n"


async def fetch_data(params, stdin=None):
    await asyncio.sleep(0)  # simulate async I/O
    key = params.get("key", "default")
    return f"data:{key}\n"


tool = ScriptedTool("jupyter-test")
tool.add_tool(
    "reverse",
    "Reverse a string",
    callback=reverse,
    schema={"type": "object", "properties": {"text": {"type": "string"}}},
)
tool.add_tool(
    "fetch_data",
    "Fetch data by key",
    callback=fetch_data,
    schema={"type": "object", "properties": {"key": {"type": "string"}}},
)

r = tool.execute_sync("reverse --text abcdef")
assert r.exit_code == 0
assert r.stdout.strip() == "fedcba"
print("ScriptedTool sync callback OK:", repr(r.stdout.strip()))

r = tool.execute_sync("fetch_data --key mykey")
assert r.exit_code == 0
assert r.stdout.strip() == "data:mykey"
print("ScriptedTool async callback via execute_sync OK:", repr(r.stdout.strip()))

ScriptedTool sync callback OK: 'fedcba'
ScriptedTool async callback via execute_sync OK: 'data:mykey'


## 10. ScriptedTool — async callback via await execute() in Jupyter

In [11]:
r = await tool.execute("fetch_data --key asynckey")
assert r.exit_code == 0
assert r.stdout.strip() == "data:asynckey"
print("ScriptedTool async callback via await execute OK:", repr(r.stdout.strip()))

ScriptedTool async callback via await execute OK: 'data:asynckey'


## 11. Bash pipeline with custom builtin

Custom builtins participate in shell pipelines.

In [12]:
def tag(ctx: BuiltinContext) -> str:
    label = ctx.argv[0] if ctx.argv else "tag"
    text = ctx.stdin or ""
    return "\n".join(f"[{label}] {line}" for line in text.splitlines()) + "\n"


bash = Bash(custom_builtins={"tag": tag})
result = bash.execute_sync("printf 'line1\nline2\nline3\n' | tag INFO")
assert result.exit_code == 0
assert "[INFO] line1" in result.stdout
assert "[INFO] line3" in result.stdout
print("Pipeline with custom builtin OK:")
print(result.stdout.strip())

Pipeline with custom builtin OK:
[INFO] line1
[INFO] line2
[INFO] line3


## 12. Persistent state across cells

VFS and shell variables persist across `execute_sync` calls on the same `Bash` instance.

In [13]:
bash = Bash()
bash.execute_sync("export MYVAR=hello")
result = bash.execute_sync("echo $MYVAR")
assert result.exit_code == 0
assert result.stdout.strip() == "hello"
print("Shell state persists across calls OK:", repr(result.stdout.strip()))

Shell state persists across calls OK: 'hello'


## Summary

All cells passed. Key findings:
- `execute_sync()` creates its own private event loop — no conflict with Jupyter's loop
- `await execute()` runs on Jupyter's caller loop; async callbacks see the same loop
- `custom_builtins` (sync and async, BuiltinContext) work in both execution modes
- ContextVar propagation into callbacks works from Jupyter cell scope
- `ScriptedTool` works as expected with sync and async callbacks